In [1]:
from EBBI.qecc.surface_code import SurfaceCode
from EBBI.qecc.noise_models import get_noise_model
import numpy as np
import pandas as pd
from tqdm import tqdm

# Deformation index overview

- 0 = $I/(p_X,p_Y,p_Z)$
- 1 = $\sqrt{Z}/(p_Y,p_X,p_Z)$
- 2 = $H\sqrt{Z}H/(p_X,p_Z,p_Y)$
- 3 = $H/(p_Z,p_Y,p_X)$
- 4 = $\sqrt{Z}H/(p_Z,p_X,p_Y)$
- 5 = $H\sqrt{Z}/(p_Y,p_Z,p_X)$

# Hypothesis

$I\!\sim\!\sqrt{Z}$, $H\!\sim\!\sqrt{Z}H$, and $H\sqrt{Z}\!\sim\!H\sqrt{Z}H$

giving:

$0\sim1$, $2\sim5$, and $3\sim4$

In [2]:
EQUIV = { # These changes should not change the DEM
    0: 1,
    1: 0,
    2: 5,
    3: 4,
    4: 3,
    5: 2
}
CONTROL = { # This one should fail the test as the cd relations are not equivalent
    0: 1,
    1: 2, # This is the only difference between EQUIV and CONTROL
    2: 5,
    3: 4,
    4: 3,
    5: 2
}
# Construct the surface codes with the specified parameters
L = 3
NUM_TRIALS = 1000
code_x = SurfaceCode(
    distance=L,
    basis = "X",
    noise_model = get_noise_model(
        "MHBD_noise",
        error_probability = 0.001,
        bias = 500
    )[0]
)
code_z = SurfaceCode(
    distance=L,
    basis = "Z",
    noise_model = get_noise_model(
        "MHBD_noise",
        error_probability = 0.001,
        bias = 500
    )[0]
)

In [91]:
def run_test(code_x: SurfaceCode, code_z: SurfaceCode, rounds: int, num_trials: int, equiv: dict):
    """
    This check is the exchanges listed in `equiv` have any impact on the LER of the code.
    This is done decoder independently by comparing the underlying detector error models (DEMs) of the two Clifford deformations. 
    If the DEMs are identical, then the LER is guaranteed to be identical as well.
    """
    data = {
        "base_cd": [],
        "equiv_cd": [],
        "identical_x": [],
        "identical_z": []
    }
    for _ in tqdm(range(num_trials), desc="Generating trials", ncols=150):
        # Generate random base Clifford deformation
        base_cd = np.random.randint(0, 6, size=code_x.num_data_qubits)
        # Generate equivalent Clifford deformation by randomly applying the equivalence relation
        equiv_cd = np.where(np.random.randint(0, 2, size=code_x.num_data_qubits), base_cd, [equiv[i] for i in base_cd])
        # Get DEMs
        base_dem_x = code_x.stim_circuit_level(base_cd, rounds).detector_error_model(approximate_disjoint_errors=True)
        base_dem_z = code_z.stim_circuit_level(base_cd, rounds).detector_error_model(approximate_disjoint_errors=True)
        equiv_dem_x = code_x.stim_circuit_level(equiv_cd, rounds).detector_error_model(approximate_disjoint_errors=True)
        equiv_dem_z = code_z.stim_circuit_level(equiv_cd, rounds).detector_error_model(approximate_disjoint_errors=True)
        # Check if the two DEMs are identical
        identical_x = base_dem_x.approx_equals(equiv_dem_x, atol=1e-15)
        identical_z = base_dem_z.approx_equals(equiv_dem_z, atol=1e-15)
        # Store results
        data["base_cd"].append(''.join(map(str, base_cd)))
        data["equiv_cd"].append(''.join(map(str, equiv_cd)))
        data["identical_x"].append(identical_x)
        data["identical_z"].append(identical_z)

    df = pd.DataFrame(data)
    return df

# This should pass all tests as the equivalence relation is valid
print("Testing equivalence relation on the valid equivalence relations...")
df = run_test(code_x, code_z, L, NUM_TRIALS, EQUIV)
px = df['identical_x'].mean()
pz = df['identical_z'].mean()
print(f"X: Success rate: {px:.2%} ± {np.sqrt(px * (1 - px) / NUM_TRIALS):.2%} ({df['identical_x'].sum()}/{NUM_TRIALS})")
print(f"Z: Success rate: {pz:.2%} ± {np.sqrt(pz * (1 - pz) / NUM_TRIALS):.2%} ({df['identical_z'].sum()}/{NUM_TRIALS})")
# This should fail in most cases (there will be a few cases where no iligal exchanges are made and the test will pass)
print("Testing equivalence relation on the control equivalence relations...")
df = run_test(code_x, code_z, L, NUM_TRIALS, CONTROL)
px = df['identical_x'].mean()
pz = df['identical_z'].mean()
print(f"X: Success rate: {px:.2%} ± {np.sqrt(px * (1 - px) / NUM_TRIALS):.2%} ({df['identical_x'].sum()}/{NUM_TRIALS})")
print(f"Z: Success rate: {pz:.2%} ± {np.sqrt(pz * (1 - pz) / NUM_TRIALS):.2%} ({df['identical_z'].sum()}/{NUM_TRIALS})")
predicted_success_rate = (1 - (1/6) * 0.5) ** code_x.num_data_qubits
print(f"Predicted success rate: (1 - (1/num_wrong_exchanges) * per_qubit_change_probability) ** {code_x.num_data_qubits} = (1 - (1/6) * 0.5) ** {code_x.num_data_qubits} = {predicted_success_rate:.2%}")

Testing equivalence relation on the valid equivalence relations...


Generating trials: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:09<00:00, 14.38it/s]


X: Success rate: 100.00% ± 0.00% (1000/1000)
Z: Success rate: 100.00% ± 0.00% (1000/1000)
Testing equivalence relation on the control equivalence relations...


Generating trials: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:20<00:00, 12.35it/s]

X: Success rate: 45.10% ± 1.57% (451/1000)
Z: Success rate: 45.10% ± 1.57% (451/1000)
Predicted success rate: (1 - (1/num_wrong_exchanges) * per_qubit_change_probability) ** 9 = (1 - (1/6) * 0.5) ** 9 = 45.70%


Practical fault-tolerant quantum computing requires qubit-efficient quantum error correction (QEC). Clifford deformations (CDs) modify the stabilizers of a QEC code to lower its logical error rate at no additional resource cost. However, the number of possible CDs grows exponentially with the number of qubits, making exhaustive search infeasible beyond small codes. In this work, we further show that under circuit level noise the optimal CD depends strongly on both the noise model and decoder. To make systematic CD optimization possible, we introduce Evolution-Based Best-arm Identification (EBBI), a shot-efficient search algorithm that treats the circuit, noise model and decoder as a black box. We benchmark EBBI’s performance on the rotated surface code,  under two strongly $Z$-biased circuit-level noise models with the PyMatching decoder. At distance 3, EBBI finds the optimal CD using up to $\sim 1100\times$ fewer shots than random search. At distance 5, when run at the target error rate, the median EBBI run converges to the CDs we believe to be optimal, while random search never converges. Under a $Z$-biased SI1000-like noise model, EBBI discovers a new CD, the B-XZZX code, which we generalized to larger distances. It differs from XZZX on only a fraction $\mathcal{O}(d^{-1})$ of the qubits at code distance $d$, yet its relative advantage grows from $\approx 5\%$ at $d = 3$ to $\approx 12\%$ at $d = 11$. We thus show that the B-XZZX pattern found by EBBI can be systematically scaled to larger code distances. Our work represents a first step towards scalable CD optimization for general QEC codes.